# 06 — TTS (ElevenLabs v3) + Voice Cloning (IVC)

**Purpose:** Synthesise English speech for each translated segment using **character-specific cloned voices**. Voice identity preservation is non-negotiable — the English output must sound like the same actor, not a generic voice.

## Voice cloning (IVC) — why this is mandatory

When a Tamil actor speaks in the source video, the English output MUST sound like THE SAME PERSON speaking English. Generic library voices are not acceptable for any production output. We create an **Instant Voice Clone (IVC)** from clean source audio per character using the ElevenLabs `/v1/voices/add` API.

### IVC creation process
1. Load diarization output → get speaker labels + timestamps
2. For each speaker, extract their cleanest segments from `stems/vocals.wav` (longest first, up to 60s)
3. Upload to ElevenLabs IVC API → receive a `voice_id` per character
4. Cache the mapping: `tts_output/voice_map.json` → `{speaker_label: voice_id}`
5. In synthesis loop: `voice_id = speaker_voice_map[seg['speaker']]`

Speakers with < 5s of extractable clean audio fall back to `ELEVENLABS_VOICE_ID` (library voice) and are flagged for manual intervention.

## API settings rationale
- **stability 0.5**: balanced between consistent delivery and natural variation
- **similarity_boost 0.75**: strong voice cloning fidelity while retaining expressiveness
- **style 0**: no additional style exaggeration — let the audio tags drive emotion
- **pcm_44100**: lossless PCM, no mp3 artifacts — requires Pro plan
- **previous_text / next_text**: passes surrounding context so prosody flows naturally across segment boundaries

## Quality checks
- **Silence trimming**: ElevenLabs pads 100–300 ms of trailing silence; we trim at −60 dBFS to avoid duration inflation
- **Duration verification**: actual vs. target; worst offenders logged
- **WER round-trip**: ASR the TTS output → verify it matches translated text (catches mispronunciations and tag leakage)
- **Accent leak log**: flag segments where source-language accent is audible in English output

**API keys needed:** `ELEVENLABS_API_KEY`. `ELEVENLABS_VOICE_ID` is a fallback only (used when IVC fails for a speaker). See `API_KEYS.md`.

**Input:** `translation/translation.json` · `diarization/diarization.json` · `stems/vocals.wav`
**Output:** `tts_output/voice_map.json` · `tts_output/segment_{N:04d}.wav` · `tts_output/tts_manifest.json`

In [ ]:
!pip install -q elevenlabs soundfile librosa tqdm pandas openai-whisper
print('Ready.')


In [ ]:
import sys, os, json, time, requests
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
from getpass import getpass
import numpy as np
import soundfile as sf
import librosa
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import Audio, display

sys.path.insert(0, os.path.abspath('..'))
from config import (
    TRANSLATION_DIR, TTS_OUTPUT_DIR, DIARIZATION_DIR, STEMS_DIR,
    TTS_OUTPUT_SR, ELEVENLABS_MODEL_ID, ELEVENLABS_TTS_SETTINGS,
    VOICE_CLONING_TARGET_SECS, VOICE_CLONING_MIN_SECS, VOICE_CLONING_MIN_SEG_SECS,
)

TRANSLATION_JSON = os.path.join(TRANSLATION_DIR,  'translation.json')
MANIFEST_JSON    = os.path.join(TTS_OUTPUT_DIR,    'tts_manifest.json')
VOICE_MAP_JSON   = os.path.join(TTS_OUTPUT_DIR,    'voice_map.json')
DIAR_JSON        = os.path.join(DIARIZATION_DIR,   'diarization.json')
VOCALS_WAV       = os.path.join(STEMS_DIR,         'vocals.wav')

with open(TRANSLATION_JSON, encoding='utf-8') as f:
    segments = json.load(f)
print(f'Loaded {len(segments)} translated segments')

with open(DIAR_JSON) as f:
    diar_segs = json.load(f)
speakers = sorted(set(s['speaker'] for s in diar_segs))
print(f'Speakers from diarization: {speakers}')

In [ ]:
# API keys — set as env vars in .envrc, or paste when prompted.
# ELEVENLABS_VOICE_ID: fallback library voice used when IVC creation fails for a speaker.
# Find a voice ID at https://api.elevenlabs.io/v1/voices or from the ElevenLabs web app.
ELEVENLABS_API_KEY  = os.getenv('ELEVENLABS_API_KEY')  or getpass('ElevenLabs API key: ')
ELEVENLABS_VOICE_ID = os.getenv('ELEVENLABS_VOICE_ID') or getpass('ElevenLabs fallback Voice ID (library voice): ')

print(f'Model     : {ELEVENLABS_MODEL_ID}')
print(f'Fallback V: {ELEVENLABS_VOICE_ID}  (used only if IVC fails for a speaker)')
print(f'Output SR : {TTS_OUTPUT_SR} Hz (PCM)')

In [ ]:
from elevenlabs.client import ElevenLabs
from elevenlabs import VoiceSettings

client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

# Connectivity check
try:
    voices = client.voices.get_all()
    print(f'Connected. {len(voices.voices)} voices available.')
    # Verify requested voice exists
    voice_ids = [v.voice_id for v in voices.voices]
    if ELEVENLABS_VOICE_ID in voice_ids:
        chosen = next(v for v in voices.voices if v.voice_id == ELEVENLABS_VOICE_ID)
        print(f'Voice found: "{chosen.name}" ({chosen.category})')
    else:
        print(f'WARNING: Voice ID "{ELEVENLABS_VOICE_ID}" not in your library. Check the ID.')
except Exception as e:
    print(f'Connection error: {e}')


## Voice Extraction + IVC Creation

Extract the longest clean segments per speaker from the separated vocals, upload to ElevenLabs `/v1/voices/add`, and build a `speaker → voice_id` map. Cached after first run — delete `tts_output/voice_map.json` to re-clone.

In [ ]:
if os.path.exists(VOICE_MAP_JSON):
    with open(VOICE_MAP_JSON) as f:
        speaker_voice_map = json.load(f)
    print(f'Loaded cached voice map ({len(speaker_voice_map)} speakers):')
    for spk, vid in speaker_voice_map.items():
        label = '(library fallback)' if vid == ELEVENLABS_VOICE_ID else '(IVC cloned)'
        print(f'  {spk}: {vid}  {label}')
else:
    print('Creating Instant Voice Clones per speaker from stems/vocals.wav...')
    y_vocals, sr_voc = librosa.load(VOCALS_WAV, sr=None, mono=True)
    speaker_voice_map = {}

    for speaker in tqdm(speakers, desc='Creating IVCs'):
        # Gather the longest clean segments for this speaker
        spk_segs = [s for s in diar_segs
                    if s['speaker'] == speaker
                    and s['end'] - s['start'] >= VOICE_CLONING_MIN_SEG_SECS]
        spk_segs_sorted = sorted(spk_segs, key=lambda s: s['end'] - s['start'], reverse=True)

        ref_chunks, total_dur = [], 0.0
        for seg in spk_segs_sorted:
            s_idx = int(seg['start'] * sr_voc)
            e_idx = int(seg['end']   * sr_voc)
            ref_chunks.append(y_vocals[s_idx:e_idx])
            total_dur += seg['end'] - seg['start']
            if total_dur >= VOICE_CLONING_TARGET_SECS:
                break

        if total_dur < VOICE_CLONING_MIN_SECS:
            print(f'  {speaker}: only {total_dur:.1f}s clean audio — using fallback library voice')
            speaker_voice_map[speaker] = ELEVENLABS_VOICE_ID
            continue

        # Save reference audio
        ref_audio = np.concatenate(ref_chunks)
        ref_path  = os.path.join(TTS_OUTPUT_DIR, f'ref_{speaker}.wav')
        sf.write(ref_path, ref_audio, sr_voc, subtype='PCM_16')

        # Upload to ElevenLabs IVC
        try:
            with open(ref_path, 'rb') as ref_file:
                resp = requests.post(
                    'https://api.elevenlabs.io/v1/voices/add',
                    headers={'xi-api-key': ELEVENLABS_API_KEY},
                    files={'files': (f'ref_{speaker}.wav', ref_file, 'audio/wav')},
                    data={
                        'name': f'{speaker}_clone',
                        'description': f'Auto-cloned — {speaker} ({total_dur:.0f}s reference)',
                    },
                )
            resp.raise_for_status()
            clone_id = resp.json()['voice_id']
            speaker_voice_map[speaker] = clone_id
            print(f'  {speaker}: IVC created -> {clone_id}  ({total_dur:.1f}s reference)')
        except Exception as e:
            print(f'  {speaker}: IVC FAILED ({e}) — using fallback voice')
            speaker_voice_map[speaker] = ELEVENLABS_VOICE_ID

    with open(VOICE_MAP_JSON, 'w') as f:
        json.dump(speaker_voice_map, f, indent=2)
    print(f'Voice map saved -> {VOICE_MAP_JSON}')

n_cloned   = sum(1 for v in speaker_voice_map.values() if v != ELEVENLABS_VOICE_ID)
n_fallback = len(speaker_voice_map) - n_cloned
print(f'\nSummary: {n_cloned} cloned voices, {n_fallback} fallback (library voice)')
if n_fallback:
    fb_spks = [k for k, v in speaker_voice_map.items() if v == ELEVENLABS_VOICE_ID]
    print(f'Fallback speakers (need manual reference audio or longer diarization): {fb_spks}')

In [ ]:
def synthesise_segment(text, voice_id, prev_text='', next_text='', output_path=None):
    '''ElevenLabs v3 TTS -> float32 numpy array + actual duration.

    PCM int16 divisor: use np.iinfo(np.int16).max (32767) for correct range.
    '''
    response = client.text_to_speech.convert(
        voice_id=voice_id,
        text=text,
        model_id=ELEVENLABS_MODEL_ID,
        voice_settings=VoiceSettings(
            stability=ELEVENLABS_TTS_SETTINGS['stability'],
            similarity_boost=ELEVENLABS_TTS_SETTINGS['similarity_boost'],
            style=ELEVENLABS_TTS_SETTINGS['style'],
            use_speaker_boost=True,
        ),
        previous_text=prev_text or None,
        next_text=next_text     or None,
        output_format='pcm_44100',
    )
    pcm_bytes  = b''.join(response)
    audio_i16  = np.frombuffer(pcm_bytes, dtype=np.int16)
    audio_f32  = audio_i16.astype(np.float32) / np.iinfo(np.int16).max  # FIX: 32767 not 32768

    # ── Trim trailing silence added by ElevenLabs ──────────────────────────────
    # ElevenLabs pads 100–300 ms of silence at the end; trim below -60 dBFS.
    SILENCE_THRESH = 10 ** (-60 / 20)  # -60 dBFS linear
    non_silent     = np.where(np.abs(audio_f32) > SILENCE_THRESH)[0]
    if len(non_silent) > 0:
        audio_f32 = audio_f32[:non_silent[-1] + 1]

    actual_duration = len(audio_f32) / TTS_OUTPUT_SR

    if output_path:
        sf.write(output_path, audio_f32, TTS_OUTPUT_SR, subtype='PCM_16')

    return audio_f32, actual_duration

print('TTS function ready (with silence trimming).')


In [ ]:
# Load voice map if cell-voice-clone was skipped (re-run safety)
if 'speaker_voice_map' not in dir():
    if os.path.exists(VOICE_MAP_JSON):
        with open(VOICE_MAP_JSON) as f:
            speaker_voice_map = json.load(f)
        print(f'Loaded voice map from disk: {len(speaker_voice_map)} speakers')
    else:
        print('WARNING: No voice map found. Run the IVC creation cell first.')
        print('Falling back to library voice for all speakers.')
        speaker_voice_map = {}

manifest = []
errors   = []

t0_total = time.time()
for i, seg in enumerate(tqdm(segments, desc='ElevenLabs v3 TTS')):
    out_path = os.path.join(TTS_OUTPUT_DIR, f'segment_{i:04d}.wav')

    # Per-speaker cloned voice; library voice as fallback
    voice_id = speaker_voice_map.get(seg['speaker'], ELEVENLABS_VOICE_ID)

    if os.path.exists(out_path):
        actual_dur = librosa.get_duration(path=out_path)
    else:
        text      = seg['translated_text']
        prev_text = segments[i-1]['translated_text'] if i > 0              else ''
        next_text = segments[i+1]['translated_text'] if i < len(segments)-1 else ''

        for attempt in range(3):
            try:
                _, actual_dur = synthesise_segment(text, voice_id, prev_text, next_text, out_path)
                break
            except Exception as e:
                if attempt == 2:
                    errors.append({'index': i, 'error': str(e)})
                    actual_dur = 0.0
                    break
                time.sleep(2 ** attempt)

        time.sleep(0.2)  # courtesy rate-limit pause

    target_dur = seg['target_duration']
    ratio = actual_dur / target_dur if target_dur > 0 else 1.0
    manifest.append({
        'index':           i,
        'speaker':         seg['speaker'],
        'voice_id':        voice_id,
        'voice_cloned':    voice_id != ELEVENLABS_VOICE_ID,
        'start':           seg['start'],
        'end':             seg['end'],
        'target_duration': target_dur,
        'actual_duration': round(actual_dur, 3),
        'duration_ratio':  round(ratio, 3),
        'duration_ok':     0.85 <= ratio <= 1.15,
        'path':            out_path,
        'translated_text': seg['translated_text'],
        'tags_used':       seg.get('tags_used', []),
    })

print(f'\nSynthesis complete: {len(manifest)} segs, {len(errors)} errors')
print(f'Total time: {time.time()-t0_total:.0f}s')

cloned_count = sum(1 for m in manifest if m['voice_cloned'])
print(f'Cloned voices used: {cloned_count}/{len(manifest)} segments')
if cloned_count < len(manifest):
    fallback_spks = {m['speaker'] for m in manifest if not m['voice_cloned']}
    print(f'Library voice fallback for speakers: {fallback_spks}')

with open(MANIFEST_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest -> {MANIFEST_JSON}')

In [ ]:
df = pd.DataFrame(manifest)[['index','speaker','target_duration','actual_duration','duration_ratio','duration_ok']]
fit = df['duration_ok'].sum()
print(f'Duration fit: {fit}/{len(df)} within ±15%')
print(f'Mean ratio: {df["duration_ratio"].mean():.3f}  Std: {df["duration_ratio"].std():.3f}')
print('\nWorst 10 offenders:')
worst = df.assign(abs_err=(df['duration_ratio']-1).abs()).nlargest(10, 'abs_err')
print(worst[['index','speaker','target_duration','actual_duration','duration_ratio']].to_string(index=False))


In [ ]:
# ── WER round-trip check ──────────────────────────────────────────────────────
# ASR the TTS output and compare to the original translated_text.
# Catches: mispronunciations, tag bleed (tags spoken aloud), silence-only output.
try:
    import whisper, torch, jiwer

    device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Loading Whisper base for round-trip WER on {device}...')
    wer_model = whisper.load_model('base', device=device)

    wer_scores = []
    for item in tqdm(manifest[:20], desc='WER round-trip (first 20 segs)'):  # sample
        if not os.path.exists(item['path']): continue
        res = wer_model.transcribe(item['path'], language='en')
        hyp = res['text'].strip().lower()
        ref = item['translated_text'].lower()
        # Strip audio tags from reference before WER
        import re
        ref_clean = re.sub(r'\[[^\]]+\]', '', ref).strip()
        w = jiwer.wer(ref_clean, hyp)
        wer_scores.append(w)
        if w > 0.3:
            print(f'  [{item["index"]:04d}] WER={w:.2f}  REF: {ref_clean[:60]}  HYP: {hyp[:60]}')

    print(f'\nMean round-trip WER (first 20): {np.mean(wer_scores):.3f}')
    print('(WER > 0.3 may indicate mispronunciation or tag-leak — review those segments)')
except Exception as e:
    print(f'Round-trip WER skipped: {e}')


In [ ]:
N_LISTEN = 5
for item in manifest[:N_LISTEN]:
    print(f'\n[{item["index"]:04d}] {item["speaker"]}  '
          f'target={item["target_duration"]:.2f}s  actual={item["actual_duration"]:.2f}s  '
          f'ratio={item["duration_ratio"]:.2f}  tags={item["tags_used"]}')
    print(f'  "{item["translated_text"][:100]}"')
    display(Audio(item['path']))